# 05 — Walk-Forward Evaluation

**This is the core evaluation notebook.** It answers the project's central question:

> Do our signals have **consistent, positive IC** across out-of-sample periods?

Key metrics computed here:
- Per-fold IC and ICIR
- Top-20% vs bottom-20% monthly spread
- Hit rate and stability
- Model comparison table

**Read before interpreting results:**
- IC > 0.02 consistently is a respectable result on public data
- ICIR > 0.5 suggests signal is stable enough to trust across regimes
- Hit rate > 55% suggests signal is directionally correct more often than not
- Anything materially higher deserves extra scrutiny for data leakage

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as cfg
from src.models import WalkForwardRunner
from src.targets import align_features_targets
from src.metrics import (
    rolling_ic, rolling_topk_spread, ic_summary, hit_rate, evaluate_walk_forward
)
from src.plotting import (
    plot_rolling_ic, plot_ic_comparison, plot_topk_spread,
    plot_quintile_returns, plot_feature_importance
)

PROCESSED = cfg.processed_dir()
print('Modules loaded')

## 1. Load Processed Features and Targets

In [ ]:
features = pd.read_parquet(PROCESSED / 'features.parquet')
targets  = pd.read_parquet(PROCESSED / 'targets.parquet')

X, y_all = align_features_targets(features, targets)

y_rank   = y_all['rank_target']
y_binary = y_all['top_label']
y_return = y_all['fwd_return']

print(f'Features: {X.shape[0]:,} rows × {X.shape[1]} columns')
print(f'Date range: {X.index.get_level_values("Date").min()} → {X.index.get_level_values("Date").max()}')
print(f'Tickers: {X.index.get_level_values("Ticker").nunique()}')
print(f'Target (rank) null rate: {y_rank.isna().mean():.1%}')

## 2. Walk-Forward Cross-Validation

In [ ]:
from src.validation import WalkForwardSplitter

splitter = WalkForwardSplitter(
    min_train_months = cfg.get('validation.min_train_months', 24),
    test_months      = cfg.get('validation.test_months', 3),
    gap_days         = cfg.get('validation.gap_days', 5),
    strategy         = cfg.get('validation.strategy', 'expanding'),
)

print('Walk-forward split summary:')
print(splitter.summary(X).to_string(index=False))

In [ ]:
# Run all models
runner = WalkForwardRunner(X, y_rank, y_binary=y_binary, splitter=splitter)
results = runner.run_all()

# Comparison table
comparison = WalkForwardRunner.compare_models(results)
print('\n=== Model Comparison (OOS IC) ===')
print(comparison.to_string())

## 3. IC Analysis — Best Model

In [ ]:
# Select best model by ICIR
best_name = comparison['icir'].idxmax()
best = results[best_name]
print(f'Best model by ICIR: {best_name}')
print(best.summary())

# Full IC time series
oos_scores  = best.all_oos_scores
oos_returns = best.all_oos_returns

ic_ts = rolling_ic(oos_scores, oos_returns)
fig = plot_rolling_ic(ic_ts, model_name=best_name)
plt.show()

## 4. IC Comparison — All Models

In [ ]:
ic_dict = {}
for name, res in results.items():
    sc = res.all_oos_scores
    rt = res.all_oos_returns
    ic_dict[name] = rolling_ic(sc, rt)

fig = plot_ic_comparison(ic_dict)
plt.show()

## 5. Top-K Spread Analysis

In [ ]:
spread_ts = rolling_topk_spread(oos_scores, oos_returns, top_pct=0.20, bottom_pct=0.20)

print(f'Top-20% vs Bottom-20% Spread Summary:')
print(f'  Mean spread : {spread_ts.mean()*100:.3f}%')
print(f'  Std spread  : {spread_ts.std()*100:.3f}%')
print(f'  Hit rate    : {hit_rate(spread_ts):.1%}')

fig = plot_topk_spread(spread_ts, model_name=best_name)
plt.show()

## 6. Quintile Return Profile

In [ ]:
from src.metrics import quintile_returns

q_ret = quintile_returns(oos_scores, oos_returns, n_bins=5)
print('Average return by quintile (Q1=lowest score, Q5=highest):')
for q, r in q_ret.items():
    bar = '█' * int(abs(r) * 1000)
    sign = '+' if r > 0 else ''
    print(f'  Q{q}: {sign}{r*100:.3f}% {bar}')

fig = plot_quintile_returns(q_ret, model_name=best_name)
plt.show()

## 7. Feature Importance (XGBoost)

In [ ]:
if 'xgboost' in results:
    # Average feature importance across all folds
    fold_importances = [
        f.feature_importance for f in results['xgboost'].fold_results
        if f.feature_importance is not None
    ]
    if fold_importances:
        avg_importance = pd.concat(fold_importances, axis=1).mean(axis=1).sort_values(ascending=False)
        print('Top 15 features by average XGBoost importance:')
        print(avg_importance.head(15).to_string())

        fig = plot_feature_importance(avg_importance, top_n=20, model_name='XGBoost')
        plt.show()
else:
    print('XGBoost not in results — install xgboost and re-run')

## 8. Regime Analysis — IC Conditional on VIX Level

In [ ]:
try:
    macro = pd.read_parquet(cfg.PROJECT_ROOT / cfg.get('data.raw_dir') / 'macro.parquet')
    vix = macro['vix'].reindex(ic_ts.index, method='ffill')

    high_vix = vix > 25
    low_vix  = vix <= 25

    ic_high = ic_ts[high_vix.reindex(ic_ts.index)]
    ic_low  = ic_ts[low_vix.reindex(ic_ts.index)]

    print('IC conditional on VIX regime:')
    print(f'  High VIX (>25): mean IC = {ic_high.mean():.4f} | n = {ic_high.notna().sum()}')
    print(f'  Low  VIX (≤25): mean IC = {ic_low.mean():.4f}  | n = {ic_low.notna().sum()}')
    print('')
    print('Interpretation: signal decay/improvement in high-volatility regimes')
except Exception as e:
    print(f'Macro data not available for regime analysis: {e}')

## 9. Results Summary

Fill this in after running:

| Model | Mean IC | ICIR | Hit Rate | Top-K Spread |
|-------|---------|------|----------|-------------|
| Ridge | | | | |
| Logistic | | | | |
| XGBoost | | | | |
| LightGBM | | | | |

**Key observations:**
- (Fill in after running)

**Limitations to note:**
- Survivorship bias: static ticker list
- No transaction costs in spread calculation
- Fundamental features are not point-in-time safe

→ Proceed to `06_backtest_lite.ipynb` for portfolio simulation